# Composite DNA Decoder — Training & Evaluation (Uniform-Ratio Alphabets)

**Two run modes** (set `RUN_MODE` in Cell 2):
- `"train_and_eval"` — trains a Bi-LSTM for each coverage level M, saves best weights, then evaluates all four decoders.
- `"eval_only"` — loads previously-saved `best_model_M{M}.pth` and reproduces the same train/val split, then evaluates.

**Single run can sweep one profile or all eight.** Set `ERROR_MODEL` to a name like `"EZ17"`, a list like `["EZ17","R21"]`, or the string `"ALL"`.

**Supports 3 uniform alphabet modes** (set via `ALPHABET_MODE`): `2mix_only` (10 classes), `2mix_3mix` (14 classes), `2mix_3mix_4mix` (15 classes).

**Metrics evaluated for every (profile, M, decoder) combination:**
- **NER** (Nucleotide Error Rate) — symbol-level, analogous to BER in communication.
- **SER** (Sequence Error Rate) — fraction of sequences with ≥ 1 wrong position.
- **FailureRate** — alias of SER; reported under both names for clarity in comparison.
- **Accuracy (%)** — `100·(1 − NER)`.

**Outputs** are written under `Results/<subdir>/` where `<subdir>` is derived from `(profile, alphabet, hidden_dim, num_layers, lr, eps_kl_ml, seed)` so different runs never overwrite each other. Each sub-directory contains:
- `best_model_M{M}.pth`, `final_model_M{M}.pth`, `training_history_M{M}.json`
- `split_indices_M{M}.pkl` — the deterministic train/val indices (so `eval_only` reproduces the exact split)
- `experiment_results.mat` — full numeric results for MATLAB plotting
- `experiment_results.txt` — readable text summary
- `final_comparison_plot.png` — quick-look Accuracy / NER vs M

**Reproducibility.** Each `(dataset, M)` pair gets a deterministic train/val split (SHA256 of seed + filename + M). Re-running with `RUN_MODE='eval_only'` loads the saved split and the saved best weights, so test-set numbers are identical.

In [1]:
# =============================================================================
# CELL 1: IMPORTS & DEVICE
# =============================================================================
import os
# Pin GPU if needed (uncomment & set):
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import random
import pickle
import json
import time
import hashlib
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

from scipy.io import savemat

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")


Using device: cuda
   GPU: NVIDIA GeForce RTX 3080


In [2]:
# =============================================================================
# CELL 2: CONFIGURATION
# =============================================================================

# ------------------- SELECT PROFILE(S) -------------------
# Single profile, list of profiles, or "ALL" to sweep all 8.
ERROR_MODEL = ["B22", "BOS22", "NP22", "NPF22"] # e.g. "EZ17", ["EZ17","R21"], or "ALL"

# ------------------- SELECT ALPHABET -------------------
ALPHABET_MODE = "2mix_3mix_4mix"  # "2mix_only" | "2mix_3mix" | "2mix_3mix_4mix"

# ------------------- RUN MODE -------------------
RUN_MODE = "train_and_eval"  # "train_and_eval" or "eval_only"

# ------------------- COVERAGE LEVELS -------------------
# Default per spec for uniform: [25, 15, 10, 5, 3, 1]
COVERAGE_LEVELS = [25, 15, 10, 5, 3, 1]

# ------------------- KL/ML SMOOTHING EPSILON -------------------
# Three ways to set this:
#   1. "auto"   -> use the per-profile best epsilon hard-coded in
#                  ERROR_MODEL_SPECS below (recommended; tuned per profile
#                  via average-accuracy sweep across coverage levels).
#   2. A scalar -> use the same epsilon for every profile (e.g. 0.01).
#   3. A dict mapping profile-name -> epsilon, e.g.
#        EPSILON_KLML = {"EZ17": 0.01, "R21": 0.1}
#      Any profile not in the dict falls back to the hard-coded "best_eps_klml".
EPSILON_KLML = "auto"

# ------------------- DATASET PARAMETERS (must match generator) -------------------
NUM_SAMPLES   = 100000
MAX_COVERAGE  = 50
DATASET_DIR   = "./dataset"

# ------------------- TRAINING HYPERPARAMETERS -------------------
HIDDEN_DIM     = 128
NUM_LAYERS     = 2
DROPOUT        = 0.2
BIDIRECTIONAL  = True
BATCH_SIZE     = 500
LEARNING_RATE  = 1e-3
WEIGHT_DECAY   = 1e-4
EPOCHS         = 100
PATIENCE       = 10
WARMUP_EPOCHS  = 10
MIN_LR         = 1e-6
TRAIN_FRACTION = 0.8
SEED           = 42

RESULTS_ROOT   = "./Results"

# ------------------- PROFILE SPECS -------------------
# "best_eps_klml" is the per-profile epsilon that maximises average KL/ML
# decoder accuracy across all coverage levels, determined by an offline
# sensitivity sweep. These values are hard-coded so we don't have to tune
# epsilon manually for every experiment.
ERROR_MODEL_SPECS = {
    "EZ17":  {"seq_length": 136, "name": "EZ17",  "platform": "Illumina miSeq + Twist Bioscience",
              "best_eps_klml": 0.01},
    "G15":   {"seq_length": 104, "name": "G15",   "platform": "Illumina miSeq + CustomArray",
              "best_eps_klml": 0.05},
    "O17":   {"seq_length": 77,  "name": "O17",   "platform": "Illumina NextSeq + Twist Bioscience",
              "best_eps_klml": 0.01},
    "R21":   {"seq_length": 136, "name": "R21",   "platform": "Nanopore MinION + Twist Bioscience",
              "best_eps_klml": 0.1},
    "B22":   {"seq_length": 136, "name": "B22",   "platform": "Nanopore MinION short-read + Twist",
              "best_eps_klml": 0.1},
    "BOS22": {"seq_length": 136, "name": "BOS22", "platform": "Illumina miSeq 2022 + Twist",
              "best_eps_klml": 0.001},
    "NP22":  {"seq_length": 136, "name": "NP22",  "platform": "Nanopore Pilot Nov-2022",
              "best_eps_klml": 0.1},
    "NPF22": {"seq_length": 136, "name": "NPF22", "platform": "Nanopore Full Nov-2022",
              "best_eps_klml": 0.1},
    # Backwards-compat aliases (share the canonical profile's best epsilon)
    "erlich":   {"seq_length": 136, "name": "EZ17",  "platform": "Illumina miSeq + Twist Bioscience",
                 "best_eps_klml": 0.01},
    "grass":    {"seq_length": 104, "name": "G15",   "platform": "Illumina miSeq + CustomArray",
                 "best_eps_klml": 0.05},
    "organick": {"seq_length": 77,  "name": "O17",   "platform": "Illumina NextSeq + Twist Bioscience",
                 "best_eps_klml": 0.01},
}

ALL_PROFILES = ["EZ17", "G15", "O17", "R21", "B22", "BOS22", "NP22", "NPF22"]

VOCAB_SIZES = {"2mix_only": 10, "2mix_3mix": 14, "2mix_3mix_4mix": 15}
assert ALPHABET_MODE in VOCAB_SIZES, f"Unknown ALPHABET_MODE: {ALPHABET_MODE}"

ALIASES = {"erlich": "EZ17", "grass": "G15", "organick": "O17"}
def _canon(p):
    return ALIASES.get(p, p)

def _resolve_profiles(spec):
    if isinstance(spec, str):
        if spec.upper() == "ALL":
            return list(ALL_PROFILES)
        return [_canon(spec)]
    elif isinstance(spec, (list, tuple)):
        return [_canon(s) for s in spec]
    raise TypeError(f"ERROR_MODEL must be str or list, got {type(spec)}")

PROFILES_TO_RUN = _resolve_profiles(ERROR_MODEL)
for p in PROFILES_TO_RUN:
    assert p in ERROR_MODEL_SPECS, f"Unknown profile {p}"

assert RUN_MODE in ("train_and_eval", "eval_only"), f"Unknown RUN_MODE: {RUN_MODE}"

def _resolve_epsilon_for(profile_name):
    """Resolve EPSILON_KLML for a single profile, honouring its hard-coded best."""
    specs = ERROR_MODEL_SPECS[profile_name]
    profile_best = specs.get("best_eps_klml", 0.01)
    if isinstance(EPSILON_KLML, str):
        if EPSILON_KLML.lower() == "auto":
            return profile_best
        try:
            return float(EPSILON_KLML)
        except ValueError:
            raise ValueError(f'EPSILON_KLML must be "auto", a number, or a dict; got {EPSILON_KLML!r}')
    elif isinstance(EPSILON_KLML, dict):
        return float(EPSILON_KLML.get(profile_name, profile_best))
    elif isinstance(EPSILON_KLML, (int, float)):
        return float(EPSILON_KLML)
    else:
        raise TypeError(f'EPSILON_KLML must be "auto", a number, or a dict; got {type(EPSILON_KLML)}')

# Build a per-profile CONFIG dict.
def make_config_for_profile(profile_name):
    specs = ERROR_MODEL_SPECS[profile_name]
    eps = _resolve_epsilon_for(profile_name)
    # Sub-directory key includes profile, alphabet, and key hyperparameters
    # so different (lr, hidden, eps, seed, ...) runs never overwrite each other.
    subdir = (f"{specs['name']}_{ALPHABET_MODE}"
              f"_H{HIDDEN_DIM}_L{NUM_LAYERS}_lr{LEARNING_RATE:.0e}"
              f"_eps{eps:g}_seed{SEED}")
    results_dir = os.path.join(RESULTS_ROOT, subdir)
    dataset_name = f"dna_{specs['name']}_{ALPHABET_MODE}"
    dataset_path = f"{DATASET_DIR}/{dataset_name}_{NUM_SAMPLES}_{MAX_COVERAGE}.pkl"

    cfg = {
        "error_model":   profile_name,
        "error_name":    specs["name"],
        "platform":      specs["platform"],
        "seq_length":    specs["seq_length"],
        "alphabet_mode": ALPHABET_MODE,
        "vocab_size":    VOCAB_SIZES[ALPHABET_MODE],
        "dataset_path":  dataset_path,
        "results_dir":   results_dir,
        "run_mode":      RUN_MODE,
        "coverage_levels": list(COVERAGE_LEVELS),
        # Model
        "input_channels": 4,
        "hidden_dim":     HIDDEN_DIM,
        "num_layers":     NUM_LAYERS,
        "dropout":        DROPOUT,
        "bidirectional":  BIDIRECTIONAL,
        # Training
        "batch_size":     BATCH_SIZE,
        "learning_rate":  LEARNING_RATE,
        "weight_decay":   WEIGHT_DECAY,
        "epochs":         EPOCHS,
        "patience":       PATIENCE,
        "warmup_epochs":  WARMUP_EPOCHS,
        "min_lr":         MIN_LR,
        "train_fraction": TRAIN_FRACTION,
        "epsilon_klml":   eps,
        "epsilon_klml_setting": EPSILON_KLML if not isinstance(EPSILON_KLML, dict) else dict(EPSILON_KLML),
        "epsilon_klml_source":  ("auto (per-profile best)" if (isinstance(EPSILON_KLML, str) and EPSILON_KLML.lower() == "auto")
                                 else ("dict override" if isinstance(EPSILON_KLML, dict)
                                       else "scalar override")),
        "seed":           SEED,
    }
    return cfg

os.makedirs(RESULTS_ROOT, exist_ok=True)

print('=' * 72)
print(' CONFIGURATION')
print('=' * 72)
print(f"   Run mode        : {RUN_MODE}")
if len(PROFILES_TO_RUN) == 1:
    print(f"   Mode            : SINGLE profile")
else:
    print(f"   Mode            : BATCH ({len(PROFILES_TO_RUN)} profiles)")
print(f"   Profiles        : {', '.join(PROFILES_TO_RUN)}")
print(f"   Alphabet mode   : {ALPHABET_MODE}  ({VOCAB_SIZES[ALPHABET_MODE]} classes)")
print(f"   Coverage levels : {COVERAGE_LEVELS}")
if isinstance(EPSILON_KLML, str) and EPSILON_KLML.lower() == "auto":
    print(f"   Epsilon (KL/ML) : auto (per-profile best, hard-coded in ERROR_MODEL_SPECS)")
    print(f"     Per-profile resolved values:")
    for p in PROFILES_TO_RUN:
        print(f"       {p:<6} -> eps = {_resolve_epsilon_for(p):g}")
elif isinstance(EPSILON_KLML, dict):
    print(f"   Epsilon (KL/ML) : dict override")
    for p in PROFILES_TO_RUN:
        print(f"       {p:<6} -> eps = {_resolve_epsilon_for(p):g}")
else:
    print(f"   Epsilon (KL/ML) : {EPSILON_KLML}  (scalar override, same for every profile)")
print(f"   Seed            : {SEED}")
print(f"   Results root    : {RESULTS_ROOT}/")
print()
print(f"   Planned run dirs:")
for p in PROFILES_TO_RUN:
    cfg = make_config_for_profile(p)
    print(f"     • {p:<6} -> {cfg['results_dir']}")
print('=' * 72)


 CONFIGURATION
   Run mode        : train_and_eval
   Mode            : BATCH (4 profiles)
   Profiles        : B22, BOS22, NP22, NPF22
   Alphabet mode   : 2mix_3mix_4mix  (15 classes)
   Coverage levels : [25, 15, 10, 5, 3, 1]
   Epsilon (KL/ML) : auto (per-profile best, hard-coded in ERROR_MODEL_SPECS)
     Per-profile resolved values:
       B22    -> eps = 0.1
       BOS22  -> eps = 0.001
       NP22   -> eps = 0.1
       NPF22  -> eps = 0.1
   Seed            : 42
   Results root    : ./Results/

   Planned run dirs:
     • B22    -> ./Results/B22_2mix_3mix_4mix_H128_L2_lr1e-03_eps0.1_seed42
     • BOS22  -> ./Results/BOS22_2mix_3mix_4mix_H128_L2_lr1e-03_eps0.001_seed42
     • NP22   -> ./Results/NP22_2mix_3mix_4mix_H128_L2_lr1e-03_eps0.1_seed42
     • NPF22  -> ./Results/NPF22_2mix_3mix_4mix_H128_L2_lr1e-03_eps0.1_seed42


In [3]:
# =============================================================================
# CELL 3: SEED HELPERS (deterministic everywhere)
# =============================================================================
def set_global_seed(seed):
    """Seed Python random, numpy, and torch (CPU+CUDA). Disables non-determinism."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def split_seed_for(base_seed, dataset_path, coverage_M):
    """
    Derive a deterministic seed for the train/val split from
    (base_seed, dataset filename, coverage_M).

    Using a hash of the dataset filename ensures that:
      * different datasets get different splits (so we never accidentally
        share indices between profiles),
      * the same dataset always gets the same split for a given M,
      * train_and_eval and eval_only with the same config reproduce
        the identical (train, val) partition.
    """
    key = f"{base_seed}|{os.path.basename(dataset_path)}|M{coverage_M}"
    h = hashlib.sha256(key.encode("utf-8")).digest()
    # Use first 4 bytes as unsigned 32-bit seed
    return int.from_bytes(h[:4], byteorder="big", signed=False)


print("Seed helpers ready.")


Seed helpers ready.


In [4]:
# =============================================================================
# CELL 4: SYMBOL MAPPINGS & IDEAL VECTORS  (uniform-ratio alphabets)
# =============================================================================
def build_symbol_to_idx(mode):
    s = {'A': 0, 'C': 1, 'G': 2, 'T': 3,
         'M1': 4, 'M2': 5, 'M3': 6, 'M4': 7, 'M5': 8, 'M6': 9}
    if mode in ("2mix_3mix", "2mix_3mix_4mix"):
        s.update({'T1': 10, 'T2': 11, 'T3': 12, 'T4': 13})
    if mode == "2mix_3mix_4mix":
        s.update({'Q1': 14})
    return s


def build_ideal_vectors(mode):
    vecs = [
        [1.0, 0.0, 0.0, 0.0],  # A
        [0.0, 1.0, 0.0, 0.0],  # C
        [0.0, 0.0, 1.0, 0.0],  # G
        [0.0, 0.0, 0.0, 1.0],  # T
        [0.5, 0.0, 0.0, 0.5],  # M1 A|T
        [0.0, 0.5, 0.5, 0.0],  # M2 C|G
        [0.0, 0.5, 0.0, 0.5],  # M3 C|T
        [0.0, 0.0, 0.5, 0.5],  # M4 G|T
        [0.5, 0.5, 0.0, 0.0],  # M5 A|C
        [0.5, 0.0, 0.5, 0.0],  # M6 A|G
    ]
    if mode in ("2mix_3mix", "2mix_3mix_4mix"):
        t = 1.0 / 3.0
        vecs += [[t, t, t, 0.0], [t, t, 0.0, t], [t, 0.0, t, t], [0.0, t, t, t]]
    if mode == "2mix_3mix_4mix":
        vecs.append([0.25, 0.25, 0.25, 0.25])
    return torch.tensor(vecs, dtype=torch.float32)


def build_symbol_to_idx_for_config(cfg):
    return build_symbol_to_idx(cfg['alphabet_mode'])


def build_ideal_vectors_for_config(cfg):
    return build_ideal_vectors(cfg['alphabet_mode'])


_demo = build_symbol_to_idx(ALPHABET_MODE)
print(f"Alphabet '{ALPHABET_MODE}' has {len(_demo)} symbols:")
for sym, idx in sorted(_demo.items(), key=lambda x: x[1]):
    print(f"   idx={idx:<3} {sym}")


Alphabet '2mix_3mix_4mix' has 15 symbols:
   idx=0   A
   idx=1   C
   idx=2   G
   idx=3   T
   idx=4   M1
   idx=5   M2
   idx=6   M3
   idx=7   M4
   idx=8   M5
   idx=9   M6
   idx=10  T1
   idx=11  T2
   idx=12  T3
   idx=13  T4
   idx=14  Q1


In [5]:
# =============================================================================
# CELL 6: DATA PREPROCESSING & DATASET CLASS
# =============================================================================
def preprocess_cluster_to_matrix(cluster_reads, target_length):
    """Map a cluster of variable-length noisy reads to a (4, target_length)
    normalised nucleotide-frequency matrix via linear alignment, matching the
    behaviour of the original training scripts."""
    profile_matrix = np.zeros((4, target_length), dtype=np.float32)
    base_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    num_reads = len(cluster_reads)

    for read in cluster_reads:
        read_len = len(read)
        if read_len == 0:
            continue
        for t_idx in range(target_length):
            read_idx = int((t_idx + 0.5) * (read_len / target_length))
            if read_idx >= read_len:
                read_idx = read_len - 1
            base = read[read_idx]
            if base in base_map:
                profile_matrix[base_map[base], t_idx] += 1.0

    if num_reads > 0:
        profile_matrix /= num_reads
    return profile_matrix


class CompositeDNADataset(Dataset):
    """PyTorch dataset for composite-DNA pickles produced by the
    dataset_generator notebooks."""

    def __init__(self, data_path, seq_length, symbol_to_idx, limit_coverage=None):
        with open(data_path, 'rb') as f:
            raw = pickle.load(f)
        self.samples = raw['data']
        self.metadata = raw['metadata']
        self.seq_length = seq_length
        self.symbol_to_idx = symbol_to_idx
        self.limit_coverage = limit_coverage

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        cluster = item['cluster']
        if self.limit_coverage is not None:
            actual = min(self.limit_coverage, len(cluster))
            cluster = cluster[:actual]
        x = preprocess_cluster_to_matrix(cluster, self.seq_length)
        y = np.array([self.symbol_to_idx[s] for s in item['label']], dtype=np.longlong)
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)


def make_reproducible_split(full_ds, base_seed, dataset_path, coverage_M,
                            train_frac=0.8, split_dir=None):
    """Return (train_subset, val_subset) using a deterministic split derived
    from (base_seed, dataset_path, M). Persists indices to split_dir so that
    eval_only mode loads the exact same split."""
    n = len(full_ds)
    n_train = int(train_frac * n)

    split_seed = split_seed_for(base_seed, dataset_path, coverage_M)
    split_file = None
    if split_dir is not None:
        os.makedirs(split_dir, exist_ok=True)
        split_file = os.path.join(split_dir, f"split_indices_M{coverage_M}.pkl")

    # If a saved split exists, load it (guarantees eval_only matches training).
    if split_file is not None and os.path.exists(split_file):
        with open(split_file, 'rb') as f:
            saved = pickle.load(f)
        if saved['seed'] == split_seed and saved['n'] == n:
            train_idx = saved['train_idx']
            val_idx = saved['val_idx']
            print(f"      Loaded saved train/val split from {split_file}")
            return Subset(full_ds, train_idx), Subset(full_ds, val_idx), split_seed

    # Otherwise compute and persist.
    rng = np.random.RandomState(split_seed)
    perm = rng.permutation(n)
    train_idx = perm[:n_train].tolist()
    val_idx = perm[n_train:].tolist()

    if split_file is not None:
        with open(split_file, 'wb') as f:
            pickle.dump({'seed': split_seed, 'n': n,
                         'train_idx': train_idx, 'val_idx': val_idx,
                         'base_seed': base_seed, 'coverage_M': coverage_M,
                         'dataset_path': dataset_path}, f)
        print(f"      Saved train/val split to {split_file}")

    return Subset(full_ds, train_idx), Subset(full_ds, val_idx), split_seed


In [6]:
# =============================================================================
# CELL 7: BI-LSTM MODEL
# =============================================================================
class CompositeDecoderLSTM(nn.Module):
    """Bi-LSTM decoder.
    Input  : (B, 4, L)   normalised frequency matrix
    Output : (B, V, L)   per-position class logits
    """

    def __init__(self, config):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=config['input_channels'],
            hidden_size=config['hidden_dim'],
            num_layers=config['num_layers'],
            batch_first=True,
            bidirectional=config['bidirectional'],
            dropout=config['dropout'] if config['num_layers'] > 1 else 0.0,
        )
        fc_in = config['hidden_dim'] * 2 if config['bidirectional'] else config['hidden_dim']
        self.fc = nn.Linear(fc_in, config['vocab_size'])

    def forward(self, x):
        x = x.permute(0, 2, 1)            # (B, L, 4)
        out, _ = self.lstm(x)             # (B, L, H')
        logits = self.fc(out)             # (B, L, V)
        return logits.permute(0, 2, 1)    # (B, V, L)


In [7]:
# =============================================================================
# CELL 8: BASELINE DECODERS  (KL / ML use a configurable smoothing epsilon)
# =============================================================================
def min_distance_decoder(obs, ideal_vectors):
    """Min-Euclidean-distance decoder. obs:(B,L,4), ideal:(C,4)."""
    dists = torch.sum(
        (obs.unsqueeze(2) - ideal_vectors.unsqueeze(0).unsqueeze(0)) ** 2, dim=3
    )
    return torch.argmin(dists, dim=2)


def kl_divergence_decoder(obs, ideal_vectors, epsilon):
    """KL-divergence decoder (minimises cross-entropy with smoothed ideals)."""
    ideal_safe = ideal_vectors.clone()
    ideal_safe = torch.clamp(ideal_safe, min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    obs_expanded = obs.unsqueeze(2)                          # (B, L, 1, 4)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    ce = -(obs_expanded * log_ideal).sum(dim=-1)             # (B, L, C)
    return torch.argmin(ce, dim=-1)


def maximum_likelihood_decoder(obs, ideal_vectors, epsilon):
    """ML decoder (maximises log-likelihood with smoothed ideals)."""
    ideal_safe = ideal_vectors.clone()
    ideal_safe = torch.clamp(ideal_safe, min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    obs_expanded = obs.unsqueeze(2)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    ll = (obs_expanded * log_ideal).sum(dim=-1)
    return torch.argmax(ll, dim=-1)


In [8]:
# =============================================================================
# CELL 9: EARLY STOPPING & TRAINING LOOP
# =============================================================================
class EarlyStopping:
    def __init__(self, patience=5, path='checkpoint.pt', verbose=True):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.path = path
        self.verbose = verbose
        self.best_val_loss = float('inf')

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None or score > self.best_score:
            if self.verbose and self.best_val_loss != float('inf'):
                print(f"      Val loss improved ({self.best_val_loss:.4f} -> {val_loss:.4f}). Saving.")
            self.best_score = score
            torch.save(model.state_dict(), self.path)
            self.best_val_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.verbose:
                print(f"      EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True


def train_model(model, train_loader, val_loader, config, weights_path, device):
    """Standard training with linear warmup + cosine annealing, early stopping."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(),
                            lr=config['learning_rate'],
                            weight_decay=config['weight_decay'])
    warmup = LinearLR(optimizer, start_factor=0.1, total_iters=config['warmup_epochs'])
    # T_max is the number of post-warmup (cosine) epochs. Guard against
    # epochs <= warmup_epochs (e.g. tiny smoke-test configs), which would make
    # CosineAnnealingLR divide by zero.
    _t_max = max(1, config['epochs'] - config['warmup_epochs'])
    cosine = CosineAnnealingLR(optimizer,
                               T_max=_t_max,
                               eta_min=config['min_lr'])
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine],
                             milestones=[config['warmup_epochs']])
    stopper = EarlyStopping(patience=config['patience'], path=weights_path, verbose=True)
    history = {'train_loss': [], 'val_loss': [], 'lr': []}

    print(f"   Training config: epochs={config['epochs']}, patience={config['patience']}, "
          f"warmup={config['warmup_epochs']}, lr={config['learning_rate']} -> {config['min_lr']}")

    for epoch in range(config['epochs']):
        t0 = time.time()
        model.train()
        train_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            out = model(inputs)
            loss = criterion(out, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
        train_loss /= max(len(train_loader), 1)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                val_loss += criterion(model(inputs), labels).item()
        val_loss /= max(len(val_loader), 1)

        cur_lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['lr'].append(cur_lr)

        print(f"   Epoch {epoch+1:03d}/{config['epochs']} | "
              f"Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
              f"LR: {cur_lr:.2e} | Time: {time.time()-t0:.1f}s")

        scheduler.step()
        stopper(val_loss, model)
        if stopper.early_stop:
            print(f"   Early stopping at epoch {epoch+1}")
            break

    return history


In [9]:
# =============================================================================
# CELL 10: EVALUATION  (Nucleotide ER, Sequence ER, Failure rate, Accuracy)
# =============================================================================
def _per_sequence_errors(pred, labels):
    """Return (num_sequences, num_with_any_error, total_positions, num_position_errors).

    pred, labels: (B, L) int tensors. We count a 'sequence' as one batch row.
    """
    B, L = labels.shape
    mismatch = (pred != labels)                       # (B, L) bool
    pos_errors = mismatch.sum().item()
    seq_has_error = mismatch.any(dim=1).sum().item()  # number of sequences with >=1 error
    return B, seq_has_error, B * L, pos_errors


def evaluate_all_decoders(model, loader, ideal_vectors, device, epsilon_klml):
    """Run all four decoders on `loader` and return a dict of metrics:
        per decoder ('lstm','mindist','kl','ml') -> dict with keys
           'accuracy'       (% positions correct, == 100 - NER%)
           'NER'            (nucleotide / symbol error rate, fraction in [0,1])
           'SER'            (sequence error rate: P(any position wrong))
           'failure_rate'   (alias for SER; included as a separate key for clarity)
           'positions_correct', 'positions_total'
           'sequences_correct', 'sequences_total'
    """
    model.eval()
    keys = ['lstm', 'mindist', 'kl', 'ml']
    pos_err_total = {k: 0 for k in keys}
    pos_total     = {k: 0 for k in keys}
    seq_err_total = {k: 0 for k in keys}
    seq_total     = {k: 0 for k in keys}

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            obs = inputs.permute(0, 2, 1)  # (B, L, 4)

            # Bi-LSTM
            logits = model(inputs)
            pred_lstm = torch.argmax(logits, dim=1)        # (B, L)

            pred_mindist = min_distance_decoder(obs, ideal_vectors)
            pred_kl = kl_divergence_decoder(obs, ideal_vectors, epsilon_klml)
            pred_ml = maximum_likelihood_decoder(obs, ideal_vectors, epsilon_klml)

            for k, pred in [('lstm', pred_lstm), ('mindist', pred_mindist),
                            ('kl', pred_kl), ('ml', pred_ml)]:
                B, se, P, pe = _per_sequence_errors(pred, labels)
                seq_total[k]     += B
                seq_err_total[k] += se
                pos_total[k]     += P
                pos_err_total[k] += pe

    metrics = {}
    for k in keys:
        ner = pos_err_total[k] / max(pos_total[k], 1)
        ser = seq_err_total[k] / max(seq_total[k], 1)
        metrics[k] = {
            'accuracy':          100.0 * (1.0 - ner),
            'NER':               ner,
            'SER':               ser,
            'failure_rate':      ser,
            'positions_correct': pos_total[k] - pos_err_total[k],
            'positions_total':   pos_total[k],
            'sequences_correct': seq_total[k] - seq_err_total[k],
            'sequences_total':   seq_total[k],
        }
    return metrics


In [10]:
# =============================================================================
# CELL 11: PER-COVERAGE EXPERIMENT  (train_and_eval OR eval_only)
# =============================================================================
def run_experiment_for_coverage(coverage_M, config, symbol_to_idx, ideal_vectors,
                                device, run_mode):
    """Run one experiment for a single coverage M.

    run_mode:
      'train_and_eval' - train Bi-LSTM, save weights, then evaluate all 4 decoders.
      'eval_only'      - load best_model_M{M}.pth and evaluate only.

    Returns (metrics, history_or_None).
    """
    print(f"\n{'='*72}")
    print(f"  COVERAGE M = {coverage_M}  |  profile = {config['error_name']}"
          f"  |  mode = {run_mode}")
    print(f"  seq_length = {config['seq_length']}, alphabet = {config['alphabet_mode']}, "
          f"vocab = {config['vocab_size']}")
    print(f"{'='*72}")

    # Re-seed for reproducibility of model init, shuffling, etc.
    set_global_seed(config['seed'])

    # Build dataset (cap reads at coverage_M)
    full_ds = CompositeDNADataset(
        config['dataset_path'], config['seq_length'], symbol_to_idx,
        limit_coverage=coverage_M,
    )
    print(f"   Dataset loaded: {len(full_ds):,} samples from {config['dataset_path']}")

    # Deterministic split (same indices every time, persisted to disk)
    train_ds, val_ds, split_seed = make_reproducible_split(
        full_ds, base_seed=config['seed'],
        dataset_path=config['dataset_path'],
        coverage_M=coverage_M, train_frac=config['train_fraction'],
        split_dir=config['results_dir'],
    )
    print(f"   Train: {len(train_ds):,}  |  Val: {len(val_ds):,}  |  split_seed={split_seed}")

    # DataLoaders. For training shuffle is reseeded by set_global_seed above.
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'],
                              shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=config['batch_size'],
                            shuffle=False, num_workers=0)

    # Model
    model = CompositeDecoderLSTM(config).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Model: {config['vocab_size']}-class Bi-LSTM, {n_params:,} parameters")

    # File paths
    rdir = config['results_dir']
    os.makedirs(rdir, exist_ok=True)
    best_path = os.path.join(rdir, f"best_model_M{coverage_M}.pth")
    final_path = os.path.join(rdir, f"final_model_M{coverage_M}.pth")
    hist_path = os.path.join(rdir, f"training_history_M{coverage_M}.json")

    history = None

    if run_mode == 'train_and_eval':
        history = train_model(model, train_loader, val_loader, config, best_path, device)
        torch.save(model.state_dict(), final_path)
        with open(hist_path, 'w') as f:
            json.dump(history, f, indent=2)
        # Load BEST weights for evaluation (not final)
        model.load_state_dict(torch.load(best_path, map_location=device))

    elif run_mode == 'eval_only':
        if not os.path.exists(best_path):
            raise FileNotFoundError(
                f"eval_only requires best_model_M{coverage_M}.pth at: {best_path}\n"
                f"  Run with RUN_MODE='train_and_eval' first."
            )
        model.load_state_dict(torch.load(best_path, map_location=device))
        print(f"   Loaded best weights from {best_path}")

    else:
        raise ValueError(f"Unknown run_mode: {run_mode!r}")

    # Evaluation
    metrics = evaluate_all_decoders(model, val_loader, ideal_vectors, device,
                                    epsilon_klml=config['epsilon_klml'])
    print(f"\n   Results @ M={coverage_M}:")
    print(f"      {'Decoder':<14} {'Acc(%)':>8} {'NER':>10} {'SER':>10} {'FailRate':>10}")
    for k, name in [('lstm','Bi-LSTM'), ('mindist','Min.Dist.'),
                    ('kl','KL Div.'), ('ml','Max Lik.')]:
        m = metrics[k]
        print(f"      {name:<14} {m['accuracy']:>8.4f} {m['NER']:>10.6f} "
              f"{m['SER']:>10.6f} {m['failure_rate']:>10.6f}")

    return metrics, history


In [11]:
# =============================================================================
# CELL 12: SAVE RESULTS  ( .mat  +  .txt  +  comparison plot )
# =============================================================================
def _np(arr):
    return np.array(arr, dtype=np.float64)


def save_results_mat(results, config, mat_path):
    """Save complete numeric results to a MATLAB .mat file.

    Top-level variables saved (so MATLAB can load them cleanly):
      coverage_M           -- (1 x K) vector of M values
      decoder_names        -- 1x4 cell of names {'lstm','mindist','kl','ml'}
      Acc_*                -- (1 x K) accuracy per decoder
      NER_*                -- (1 x K) nucleotide error rate per decoder
      SER_*                -- (1 x K) sequence error rate per decoder
      FailureRate_*        -- alias for SER per decoder
      positions_total_*    -- evaluation set size in positions
      sequences_total_*    -- evaluation set size in sequences
      profile, alphabet_mode, vocab_size, seq_length, ...    (metadata)
      epsilon_klml, learning_rate, hidden_dim, num_layers, ...
    """
    out = {}
    out['coverage_M'] = _np(results['coverage'])
    decoders = ['lstm', 'mindist', 'kl', 'ml']
    out['decoder_names'] = np.array(decoders, dtype=object)

    for d in decoders:
        out[f'Acc_{d}']             = _np([m['accuracy']          for m in results['per_M'][d]])
        out[f'NER_{d}']             = _np([m['NER']               for m in results['per_M'][d]])
        out[f'SER_{d}']             = _np([m['SER']               for m in results['per_M'][d]])
        out[f'FailureRate_{d}']     = _np([m['failure_rate']      for m in results['per_M'][d]])
        out[f'positions_total_{d}'] = _np([m['positions_total']   for m in results['per_M'][d]])
        out[f'sequences_total_{d}'] = _np([m['sequences_total']   for m in results['per_M'][d]])

    # Metadata
    meta_str_keys = ['error_model', 'error_name', 'alphabet_mode', 'dataset_path',
                     'results_dir', 'run_mode', 'timestamp', 'epsilon_klml_source']
    for k in meta_str_keys:
        if k in results['config']:
            out[k] = str(results['config'][k])
    meta_num_keys = ['seq_length', 'vocab_size', 'hidden_dim', 'num_layers',
                     'dropout', 'batch_size', 'learning_rate', 'weight_decay',
                     'epochs', 'patience', 'warmup_epochs', 'min_lr',
                     'epsilon_klml', 'seed', 'train_fraction']
    for k in meta_num_keys:
        if k in results['config']:
            out[k] = results['config'][k]

    savemat(mat_path, out, do_compression=True)
    print(f"   Saved .mat results: {mat_path}")


def build_results_report(results, config):
    """Build the full human-readable experiment report as a single string.

    The exact same text is written to the .txt file AND printed at the end of
    the run, so the on-screen summary is identical to the saved report.
    """
    decoders = [('lstm','Bi-LSTM'), ('mindist','Min. Distance'),
                ('kl','KL Divergence'), ('ml','Max. Likelihood')]
    L = []  # list of lines
    L.append('='*78)
    L.append(' COMPOSITE-DNA DECODER -- EXPERIMENT REPORT')
    L.append('='*78)
    L.append('')
    L.append(f"Timestamp        : {results['config']['timestamp']}")
    L.append(f"Run mode         : {results['config']['run_mode']}")
    L.append(f"Error profile    : {results['config']['error_name']}  "
             f"({results['config'].get('platform', 'n/a')})")
    L.append(f"Alphabet mode    : {results['config']['alphabet_mode']}")
    L.append(f"Vocabulary size  : {results['config']['vocab_size']}")
    L.append(f"Sequence length  : {results['config']['seq_length']}")
    L.append(f"Dataset          : {results['config']['dataset_path']}")
    L.append('')
    L.append('Hyperparameters')
    L.append('-'*78)
    for k in ['hidden_dim', 'num_layers', 'dropout', 'bidirectional',
              'batch_size', 'learning_rate', 'weight_decay',
              'epochs', 'patience', 'warmup_epochs', 'min_lr',
              'epsilon_klml', 'epsilon_klml_source',
              'seed', 'train_fraction']:
        if k in results['config']:
            L.append(f"  {k:<22}: {results['config'][k]}")
    L.append('')
    L.append('Coverage levels evaluated')
    L.append('-'*78)
    L.append(f"  M = {results['coverage']}")
    L.append('')
    # Main results table
    L.append('Main results table (one row per coverage level)')
    L.append('-'*78)
    for d_key, d_name in decoders:
        L.append('')
        L.append(f"  Decoder: {d_name}")
        L.append(f"  {'M':>4} | {'Accuracy(%)':>12} | {'NER':>12} | {'SER':>12} | {'FailureRate':>12} "
                 f"| {'#pos':>10} | {'#seq':>10}")
        L.append('  ' + '-'*82)
        for i, M in enumerate(results['coverage']):
            m = results['per_M'][d_key][i]
            L.append(f"  {M:>4} | {m['accuracy']:>12.4f} | {m['NER']:>12.6f} | "
                     f"{m['SER']:>12.6f} | {m['failure_rate']:>12.6f} | "
                     f"{m['positions_total']:>10} | {m['sequences_total']:>10}")
    # Per-M side-by-side comparison
    L.append('')
    L.append('')
    L.append('Side-by-side comparison (accuracy %, higher is better)')
    L.append('-'*78)
    L.append(f"  {'M':>4} | {'Bi-LSTM':>10} | {'Min.Dist':>10} | {'KL Div':>10} | {'Max Lik':>10}")
    L.append('  ' + '-'*56)
    for i, M in enumerate(results['coverage']):
        ml = results['per_M']['lstm'][i]['accuracy']
        md = results['per_M']['mindist'][i]['accuracy']
        kl = results['per_M']['kl'][i]['accuracy']
        mlk = results['per_M']['ml'][i]['accuracy']
        L.append(f"  {M:>4} | {ml:>10.4f} | {md:>10.4f} | {kl:>10.4f} | {mlk:>10.4f}")
    L.append('')
    L.append('')
    L.append('Side-by-side comparison (NER, lower is better)')
    L.append('-'*78)
    L.append(f"  {'M':>4} | {'Bi-LSTM':>12} | {'Min.Dist':>12} | {'KL Div':>12} | {'Max Lik':>12}")
    L.append('  ' + '-'*64)
    for i, M in enumerate(results['coverage']):
        ml = results['per_M']['lstm'][i]['NER']
        md = results['per_M']['mindist'][i]['NER']
        kl = results['per_M']['kl'][i]['NER']
        mlk = results['per_M']['ml'][i]['NER']
        L.append(f"  {M:>4} | {ml:>12.6f} | {md:>12.6f} | {kl:>12.6f} | {mlk:>12.6f}")
    L.append('')
    L.append('')
    L.append('Side-by-side comparison (SER / Failure rate, lower is better)')
    L.append('-'*78)
    L.append(f"  {'M':>4} | {'Bi-LSTM':>12} | {'Min.Dist':>12} | {'KL Div':>12} | {'Max Lik':>12}")
    L.append('  ' + '-'*64)
    for i, M in enumerate(results['coverage']):
        ml = results['per_M']['lstm'][i]['SER']
        md = results['per_M']['mindist'][i]['SER']
        kl = results['per_M']['kl'][i]['SER']
        mlk = results['per_M']['ml'][i]['SER']
        L.append(f"  {M:>4} | {ml:>12.6f} | {md:>12.6f} | {kl:>12.6f} | {mlk:>12.6f}")
    L.append('')
    L.append('='*78)
    L.append(' END OF REPORT')
    L.append('='*78)
    return '\n'.join(L) + '\n'


def save_results_txt(results, config, txt_path):
    """Write the experiment report to a .txt file AND print it to the notebook."""
    report = build_results_report(results, config)
    with open(txt_path, 'w') as f:
        f.write(report)
    print(f"   Saved .txt report : {txt_path}")
    # Echo the full report to the notebook output so it is visible at run end.
    print()
    print(report)


def plot_comparison(results, save_path, config):
    """Two-panel figure: Accuracy vs M, NER vs M."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    Ms = results['coverage']
    decoders = [('lstm','Bi-LSTM','#2980b9'), ('mindist','Min. Distance','#7f8c8d'),
                ('kl','KL Divergence','#27ae60'), ('ml','Max. Likelihood','#c0392b')]

    for k, label, color in decoders:
        accs = [m['accuracy'] for m in results['per_M'][k]]
        ners = [m['NER']      for m in results['per_M'][k]]
        ax1.plot(Ms, accs, 'o-', lw=2.2, ms=7, color=color, label=label)
        ax2.plot(Ms, ners, 'o-', lw=2.2, ms=7, color=color, label=label)

    for ax, ylab, title in [(ax1, 'Accuracy (%)', 'Accuracy vs Coverage'),
                            (ax2, 'NER', 'Nucleotide Error Rate vs Coverage')]:
        ax.set_xlabel('Coverage Depth M', fontsize=11)
        ax.set_ylabel(ylab, fontsize=11)
        ax.set_title(f"{title} -- {config['error_name']} / {config['alphabet_mode']}",
                     fontsize=12)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)

    ax1.set_ylim(0, 105)
    ax2.set_yscale('log')
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"   Saved comparison plot: {save_path}")


In [12]:
# =============================================================================
# CELL 13: MAIN EXECUTION LOOP
# =============================================================================
batch_t0 = time.time()
batch_summary = []

for prof_idx, profile_name in enumerate(PROFILES_TO_RUN, start=1):
    cfg = make_config_for_profile(profile_name)

    print(f"\n\n{'#'*78}")
    print(f"#  [{prof_idx}/{len(PROFILES_TO_RUN)}]  Profile: {profile_name}  ({cfg['platform']})")
    print(f"#  Run mode: {RUN_MODE}")
    print(f"#  Results dir: {cfg['results_dir']}")
    print(f"#  Coverage levels: {cfg['coverage_levels']}")
    print(f"{'#'*78}")

    # Verify dataset exists.
    if not os.path.exists(cfg['dataset_path']):
        msg = (f"Dataset not found: {cfg['dataset_path']}.\n"
               f"  Run the dataset_generator notebook first with ERROR_MODEL='{profile_name}'.")
        print(f"  [ERROR] {msg}")
        batch_summary.append({'profile': profile_name, 'status': 'no dataset', 'dir': cfg['results_dir']})
        continue

    os.makedirs(cfg['results_dir'], exist_ok=True)

    # Build vocab + ideal vectors for THIS profile's alphabet (alphabet does
    # not depend on profile, but we resolve here so it lives next to cfg).
    SYMBOL_TO_IDX = build_symbol_to_idx_for_config(cfg)
    IDX_TO_SYMBOL = {v: k for k, v in SYMBOL_TO_IDX.items()}
    IDEAL_VECTORS = build_ideal_vectors_for_config(cfg).to(device)

    results = {
        'coverage': list(cfg['coverage_levels']),
        'per_M': {'lstm': [], 'mindist': [], 'kl': [], 'ml': []},
        'config': dict(cfg),
    }
    results['config']['timestamp'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    all_histories = {}

    try:
        for M in cfg['coverage_levels']:
            metrics, history = run_experiment_for_coverage(
                M, cfg, SYMBOL_TO_IDX, IDEAL_VECTORS, device, RUN_MODE,
            )
            for k in ['lstm', 'mindist', 'kl', 'ml']:
                results['per_M'][k].append(metrics[k])
            if history is not None:
                all_histories[M] = history

        # ---- save everything ----
        mat_path = os.path.join(cfg['results_dir'], 'experiment_results.mat')
        txt_path = os.path.join(cfg['results_dir'], 'experiment_results.txt')
        plot_path = os.path.join(cfg['results_dir'], 'final_comparison_plot.png')

        save_results_mat(results, cfg, mat_path)
        save_results_txt(results, cfg, txt_path)
        plot_comparison(results, plot_path, cfg)
        if all_histories:
            with open(os.path.join(cfg['results_dir'], 'all_training_histories.json'), 'w') as f:
                json.dump({str(k): v for k, v in all_histories.items()}, f, indent=2)

        batch_summary.append({'profile': profile_name, 'status': 'ok',
                              'dir': cfg['results_dir']})

    except Exception as e:
        print(f"\n[ERROR] Profile {profile_name} failed: {type(e).__name__}: {e}")
        batch_summary.append({'profile': profile_name,
                              'status': f'FAILED: {type(e).__name__}',
                              'dir': cfg['results_dir']})

# ---- final batch summary ----
print(f"\n\n{'='*78}")
print(f" BATCH SUMMARY ({len(batch_summary)} profile(s), total {time.time()-batch_t0:.1f}s)")
print(f"{'='*78}")
print(f"   {'Profile':<8} {'Status':<22}   Output directory")
print(f"   {'-'*8} {'-'*22}   {'-'*40}")
for r in batch_summary:
    print(f"   {r['profile']:<8} {r['status']:<22}   {r['dir']}")
print(f"{'='*78}")




##############################################################################
#  [1/4]  Profile: B22  (Nanopore MinION short-read + Twist)
#  Run mode: train_and_eval
#  Results dir: ./Results/B22_2mix_3mix_4mix_H128_L2_lr1e-03_eps0.1_seed42
#  Coverage levels: [25, 15, 10, 5, 3, 1]
##############################################################################

  COVERAGE M = 25  |  profile = B22  |  mode = train_and_eval
  seq_length = 136, alphabet = 2mix_3mix_4mix, vocab = 15
   Dataset loaded: 100,000 samples from ./dataset/dna_B22_2mix_3mix_4mix_100000_50.pkl
      Saved train/val split to ./Results/B22_2mix_3mix_4mix_H128_L2_lr1e-03_eps0.1_seed42/split_indices_M25.pkl
   Train: 80,000  |  Val: 20,000  |  split_seed=2527107944
   Model: 15-class Bi-LSTM, 536,335 parameters
   Training config: epochs=100, patience=10, warmup=10, lr=0.001 -> 1e-06
   Epoch 001/100 | Train: 2.6594 | Val: 2.5176 | LR: 1.00e-04 | Time: 701.6s
   Epoch 002/100 | Train: 2.1927 | Val: 1.7935 | LR: 1.90

/homes/shubham/anaconda3/envs/pytorchenv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:149: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


   Epoch 011/100 | Train: 1.0697 | Val: 1.0455 | LR: 1.00e-03 | Time: 708.8s
      Val loss improved (1.0541 -> 1.0455). Saving.
   Epoch 012/100 | Train: 1.0594 | Val: 1.0414 | LR: 1.00e-03 | Time: 713.3s
      Val loss improved (1.0455 -> 1.0414). Saving.
   Epoch 013/100 | Train: 1.0526 | Val: 1.0337 | LR: 9.99e-04 | Time: 701.2s
      Val loss improved (1.0414 -> 1.0337). Saving.
   Epoch 014/100 | Train: 1.0469 | Val: 1.0320 | LR: 9.97e-04 | Time: 702.1s
      Val loss improved (1.0337 -> 1.0320). Saving.
   Epoch 015/100 | Train: 1.0417 | Val: 1.0257 | LR: 9.95e-04 | Time: 706.0s
      Val loss improved (1.0320 -> 1.0257). Saving.
   Epoch 016/100 | Train: 1.0365 | Val: 1.0195 | LR: 9.92e-04 | Time: 712.8s
      Val loss improved (1.0257 -> 1.0195). Saving.
   Epoch 017/100 | Train: 1.0320 | Val: 1.0154 | LR: 9.89e-04 | Time: 698.3s
      Val loss improved (1.0195 -> 1.0154). Saving.
   Epoch 018/100 | Train: 1.0282 | Val: 1.0117 | LR: 9.85e-04 | Time: 729.0s
      Val loss impro

   Epoch 078/100 | Train: 0.9459 | Val: 0.9427 | LR: 1.54e-04 | Time: 707.0s
      EarlyStopping counter: 3/10
   Epoch 079/100 | Train: 0.9456 | Val: 0.9419 | LR: 1.41e-04 | Time: 706.2s
      Val loss improved (0.9424 -> 0.9419). Saving.
   Epoch 080/100 | Train: 0.9456 | Val: 0.9429 | LR: 1.29e-04 | Time: 713.3s
      EarlyStopping counter: 1/10
   Epoch 081/100 | Train: 0.9452 | Val: 0.9421 | LR: 1.18e-04 | Time: 714.9s
      EarlyStopping counter: 2/10
   Epoch 082/100 | Train: 0.9448 | Val: 0.9414 | LR: 1.07e-04 | Time: 705.4s
      Val loss improved (0.9419 -> 0.9414). Saving.
   Epoch 083/100 | Train: 0.9447 | Val: 0.9414 | LR: 9.64e-05 | Time: 712.7s
      Val loss improved (0.9414 -> 0.9414). Saving.
   Epoch 084/100 | Train: 0.9446 | Val: 0.9413 | LR: 8.64e-05 | Time: 736.1s
      Val loss improved (0.9414 -> 0.9413). Saving.
   Epoch 085/100 | Train: 0.9444 | Val: 0.9409 | LR: 7.69e-05 | Time: 762.1s
      Val loss improved (0.9413 -> 0.9409). Saving.
   Epoch 086/100 | Tra

   Epoch 036/100 | Train: 1.2524 | Val: 1.2447 | LR: 8.22e-04 | Time: 436.1s
      Val loss improved (1.2463 -> 1.2447). Saving.
   Epoch 037/100 | Train: 1.2514 | Val: 1.2438 | LR: 8.08e-04 | Time: 429.0s
      Val loss improved (1.2447 -> 1.2438). Saving.
   Epoch 038/100 | Train: 1.2499 | Val: 1.2432 | LR: 7.94e-04 | Time: 439.6s
      Val loss improved (1.2438 -> 1.2432). Saving.
   Epoch 039/100 | Train: 1.2488 | Val: 1.2457 | LR: 7.80e-04 | Time: 435.0s
      EarlyStopping counter: 1/10
   Epoch 040/100 | Train: 1.2477 | Val: 1.2418 | LR: 7.65e-04 | Time: 441.7s
      Val loss improved (1.2432 -> 1.2418). Saving.
   Epoch 041/100 | Train: 1.2461 | Val: 1.2408 | LR: 7.50e-04 | Time: 446.2s
      Val loss improved (1.2418 -> 1.2408). Saving.
   Epoch 042/100 | Train: 1.2451 | Val: 1.2405 | LR: 7.35e-04 | Time: 442.9s
      Val loss improved (1.2408 -> 1.2405). Saving.
   Epoch 043/100 | Train: 1.2437 | Val: 1.2394 | LR: 7.19e-04 | Time: 438.3s
      Val loss improved (1.2405 -> 1.2

...

   Epoch 006/100 | Train: 2.0014 | Val: 1.9878 | LR: 5.50e-04 | Time: 144.6s
      Val loss improved (1.9939 -> 1.9878). Saving.
   Epoch 007/100 | Train: 1.9863 | Val: 1.9679 | LR: 6.40e-04 | Time: 145.2s
      Val loss improved (1.9878 -> 1.9679). Saving.
   Epoch 008/100 | Train: 1.9758 | Val: 1.9620 | LR: 7.30e-04 | Time: 146.3s
      Val loss improved (1.9679 -> 1.9620). Saving.
   Epoch 009/100 | Train: 1.9680 | Val: 1.9581 | LR: 8.20e-04 | Time: 145.9s
      Val loss improved (1.9620 -> 1.9581). Saving.
   Epoch 010/100 | Train: 1.9642 | Val: 1.9494 | LR: 9.10e-04 | Time: 145.2s
      Val loss improved (1.9581 -> 1.9494). Saving.
   Epoch 011/100 | Train: 1.9594 | Val: 1.9488 | LR: 1.00e-03 | Time: 146.2s
      Val loss improved (1.9494 -> 1.9488). Saving.
   Epoch 012/100 | Train: 1.9543 | Val: 1.9431 | LR: 1.00e-03 | Time: 145.3s
      Val loss improved (1.9488 -> 1.9431). Saving.
   Epoch 013/100 | Train: 1.9512 | Val: 1.9439 | LR: 9.99e-04 | Time: 146.7s
      EarlyStopping 

   Epoch 073/100 | Train: 1.9077 | Val: 1.9077 | LR: 2.21e-04 | Time: 144.4s
      Val loss improved (1.9080 -> 1.9077). Saving.
   Epoch 074/100 | Train: 1.9075 | Val: 1.9077 | LR: 2.07e-04 | Time: 144.9s
      Val loss improved (1.9077 -> 1.9077). Saving.
   Epoch 075/100 | Train: 1.9071 | Val: 1.9076 | LR: 1.93e-04 | Time: 145.5s
      Val loss improved (1.9077 -> 1.9076). Saving.
   Epoch 076/100 | Train: 1.9070 | Val: 1.9073 | LR: 1.79e-04 | Time: 145.8s
      Val loss improved (1.9076 -> 1.9073). Saving.
   Epoch 077/100 | Train: 1.9069 | Val: 1.9073 | LR: 1.66e-04 | Time: 147.0s
      Val loss improved (1.9073 -> 1.9073). Saving.
   Epoch 078/100 | Train: 1.9069 | Val: 1.9078 | LR: 1.54e-04 | Time: 145.6s
      EarlyStopping counter: 1/10
   Epoch 079/100 | Train: 1.9064 | Val: 1.9072 | LR: 1.41e-04 | Time: 145.4s
      Val loss improved (1.9073 -> 1.9072). Saving.
   Epoch 080/100 | Train: 1.9062 | Val: 1.9074 | LR: 1.29e-04 | Time: 145.5s
      EarlyStopping counter: 1/10
   E

   Epoch 033/100 | Train: 2.1235 | Val: 2.1177 | LR: 8.60e-04 | Time: 92.4s
      Val loss improved (2.1187 -> 2.1177). Saving.
   Epoch 034/100 | Train: 2.1231 | Val: 2.1174 | LR: 8.47e-04 | Time: 92.1s
      Val loss improved (2.1177 -> 2.1174). Saving.
   Epoch 035/100 | Train: 2.1222 | Val: 2.1171 | LR: 8.35e-04 | Time: 92.6s
      Val loss improved (2.1174 -> 2.1171). Saving.
   Epoch 036/100 | Train: 2.1216 | Val: 2.1177 | LR: 8.22e-04 | Time: 92.2s
      EarlyStopping counter: 1/10
   Epoch 037/100 | Train: 2.1214 | Val: 2.1162 | LR: 8.08e-04 | Time: 94.0s
      Val loss improved (2.1171 -> 2.1162). Saving.
   Epoch 038/100 | Train: 2.1206 | Val: 2.1163 | LR: 7.94e-04 | Time: 91.9s
      EarlyStopping counter: 1/10
   Epoch 039/100 | Train: 2.1201 | Val: 2.1152 | LR: 7.80e-04 | Time: 92.9s
      Val loss improved (2.1162 -> 2.1152). Saving.
   Epoch 040/100 | Train: 2.1199 | Val: 2.1159 | LR: 7.65e-04 | Time: 91.9s
      EarlyStopping counter: 1/10
   Epoch 041/100 | Train: 2.11

   Dataset loaded: 100,000 samples from ./dataset/dna_NPF22_2mix_3mix_4mix_100000_50.pkl
      Saved train/val split to ./Results/NPF22_2mix_3mix_4mix_H128_L2_lr1e-03_eps0.1_seed42/split_indices_M1.pkl
   Train: 80,000  |  Val: 20,000  |  split_seed=2962679698
   Model: 15-class Bi-LSTM, 536,335 parameters
   Training config: epochs=100, patience=10, warmup=10, lr=0.001 -> 1e-06
   Epoch 001/100 | Train: 2.6743 | Val: 2.5961 | LR: 1.00e-04 | Time: 39.2s
   Epoch 002/100 | Train: 2.5334 | Val: 2.5091 | LR: 1.90e-04 | Time: 40.0s
      Val loss improved (2.5961 -> 2.5091). Saving.
   Epoch 003/100 | Train: 2.5045 | Val: 2.4937 | LR: 2.80e-04 | Time: 39.3s
      Val loss improved (2.5091 -> 2.4937). Saving.
   Epoch 004/100 | Train: 2.4845 | Val: 2.4730 | LR: 3.70e-04 | Time: 39.9s
      Val loss improved (2.4937 -> 2.4730). Saving.
   Epoch 005/100 | Train: 2.4704 | Val: 2.4679 | LR: 4.60e-04 | Time: 38.9s
      Val loss improved (2.4730 -> 2.4679). Saving.
   Epoch 006/100 | Train: 2.46

   Saved comparison plot: ./Results/NPF22_2mix_3mix_4mix_H128_L2_lr1e-03_eps0.1_seed42/final_comparison_plot.png


 BATCH SUMMARY (4 profile(s), total 650226.2s)
   Profile  Status                   Output directory
   -------- ----------------------   ----------------------------------------
   B22      ok                       ./Results/B22_2mix_3mix_4mix_H128_L2_lr1e-03_eps0.1_seed42
   BOS22    ok                       ./Results/BOS22_2mix_3mix_4mix_H128_L2_lr1e-03_eps0.001_seed42
   NP22     ok                       ./Results/NP22_2mix_3mix_4mix_H128_L2_lr1e-03_eps0.1_seed42
   NPF22    ok                       ./Results/NPF22_2mix_3mix_4mix_H128_L2_lr1e-03_eps0.1_seed42
